In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U huggingface_hub

In [3]:
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [4]:
hf_token=userdata.get('hf_access_token')
login(hf_token)

In [5]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

model_path = "/content/drive/MyDrive/MiniClimate/models/final/dapttapt_minilm_climate"

model = AutoModelForMaskedLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.push_to_hub("Raj7722/MiniClimate")
tokenizer.push_to_hub("Raj7722/MiniClimate")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n3cfxad/model.safetensors:   6%|5         | 7.89MB /  134MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Raj7722/MiniClimate/commit/2fab5875a2a397971e30b6a2062ed351dfdf47db', commit_message='Upload tokenizer', commit_description='', oid='2fab5875a2a397971e30b6a2062ed351dfdf47db', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Raj7722/MiniClimate', endpoint='https://huggingface.co', repo_type='model', repo_id='Raj7722/MiniClimate'), pr_revision=None, pr_num=None)

In [6]:
#testing
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("Raj7722/MiniClimate")
model = AutoModelForMaskedLM.from_pretrained("Raj7722/MiniClimate")

text = "Global warming is caused by [MASK] gas emissions."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
predicted_token_id = outputs.logits[0, mask_token_index].argmax(dim=-1)
predicted_word = tokenizer.decode(predicted_token_id)

print("Predicted word:", predicted_word)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  134MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Predicted word: greenhouse


In [7]:
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained("Raj7722/MiniClimate")
model = AutoModelForMaskedLM.from_pretrained("Raj7722/MiniClimate")

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)

test_sentences = [
    "Global warming is caused by [MASK] gas emissions.",
    "The Arctic ice is [MASK] at an alarming rate.",
    "We need to switch to [MASK] energy sources like solar and wind.",
    "Rising sea levels are a direct result of [MASK] change.",
    "Deforestation contributes to [MASK] loss and habitat destruction.",
    "The Paris [MASK] aims to limit global temperature rise.",
    "Extreme [MASK] events like floods and droughts are becoming more frequent.",
    "Carbon [MASK] is a major driver of climate change.",
    "Governments must reduce their [MASK] footprint to fight climate change.",
    "Coral [MASK] are dying due to ocean acidification.",
]

for text in test_sentences:
    results = fill_mask(text, top_k=3)
    print(f"\nInput: {text}")
    for r in results:
        print(f"   → {r['token_str']}  (score: {r['score']:.3f})")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


Input: Global warming is caused by [MASK] gas emissions.
   → greenhouse  (score: 0.774)
   → fossil  (score: 0.162)
   → carbon  (score: 0.020)

Input: The Arctic ice is [MASK] at an alarming rate.
   → melting  (score: 0.423)
   → rising  (score: 0.232)
   → increasing  (score: 0.056)

Input: We need to switch to [MASK] energy sources like solar and wind.
   → renewable  (score: 0.269)
   → other  (score: 0.057)
   → clean  (score: 0.056)

Input: Rising sea levels are a direct result of [MASK] change.
   → climate  (score: 0.995)
   → weather  (score: 0.002)
   → global  (score: 0.000)

Input: Deforestation contributes to [MASK] loss and habitat destruction.
   → habitat  (score: 0.347)
   → forest  (score: 0.104)
   → population  (score: 0.045)

Input: The Paris [MASK] aims to limit global temperature rise.
   → agreement  (score: 0.433)
   → paris  (score: 0.050)
   → government  (score: 0.023)

Input: Extreme [MASK] events like floods and droughts are becoming more frequent.
   →